# S³ Spectral-Spatial Domain-Adaptive EEG — Binary Left-vs-Right MI

Fully fixed, dependency-ordered research notebook.

**Task:** Left Hand MI vs Right Hand MI

**Runs:** 4, 8, 12 only

**Exact targets:** S004, S015, S023, S029, S031, S042, S055, S071, S082, S095

The notebook uses a learnable Sinc front-end, dynamic graph spatial modeling, multi-scale temporal convolution, bidirectional GRU (the supplied implementation's `SimplifiedBiMamba`), SE attention, binary classification, subject-adversarial GRL, supervised contrastive learning, and label-free AdaBN.

The target subjects are never used for hyperparameter/epoch selection. A subject-level validation split is created only from the source subjects.

**Important:** the temporal module is technically a bidirectional GRU, not true Mamba.

## CELL 01 — Imports and environment

In [1]:
# ============================================================
# CELL 01 — IMPORTS AND ENVIRONMENT
# ============================================================

from pathlib import Path
import os
import gc
import json
import math
import random
import time
import warnings

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mne

import torch
import torch.nn as nn
import torch.nn.functional as F

from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    cohen_kappa_score,
    roc_auc_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
    roc_curve,
    auc
)

from sklearn.manifold import TSNE

warnings.filterwarnings("ignore")
mne.set_log_level("ERROR")

try:
    torch.set_float32_matmul_precision("high")
except Exception:
    pass

print("[OK] Imports loaded.")

[OK] Imports loaded.


## CELL 02 — Single source of truth for configuration

In [2]:
# ============================================================
# CELL 02 — CONFIGURATION + DEVICE
# ============================================================

SEED = 42
DATA_DIR = "./eegmmidb"

# Binary Left-vs-Right motor imagery
RUNS = [4, 8, 12]
TMIN = 0.0
TMAX = 4.0
FS = 250.0
N_CHANNELS = 22
N_CLASSES = 2
CLASS_NAMES = ["Left Hand MI", "Right Hand MI"]

# Exact requested held-out subjects
TEST_SUBJECTS = [4, 15, 23, 29, 31, 42, 55, 71, 82, 95]

TOTAL_SUBJECTS = 109
DOMAIN_CLASSES = TOTAL_SUBJECTS

# Subject-level source validation
VAL_FRACTION = 0.10
VAL_SEED = 123

# Training
BATCH_SIZE = 64
NUM_EPOCHS = 120
MIN_EPOCHS = 30
PATIENCE = 20
LR = 3e-4
MIN_LR = 1e-6
WEIGHT_DECAY = 1e-4
LABEL_SMOOTHING = 0.02
GRAD_CLIP = 1.0

# Multi-objective training
SUPCON_WEIGHT = 0.05
SUPCON_TEMP = 0.10
DOMAIN_WEIGHT_MAX = 0.08
DOMAIN_WARMUP_EPOCHS = 25
GRL_START_EPOCH = 10

# Model
NUM_FILTERS = 10
SINC_KERNEL = 81
SPATIAL_DIM = 64

# Augmentation
AUGMENT_PROB = 0.65
NOISE_STD = 0.015
AMPLITUDE_JITTER = 0.05
CHANNEL_DROPOUT = 0.05

RESULTS_DIR = Path("./results_binary_exact10_v2")
FIG_DIR = RESULTS_DIR / "figures"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    try:
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False
    except Exception:
        pass

seed_everything()

if torch.cuda.is_available():
    DEVICE = torch.device("cuda")
elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
    DEVICE = torch.device("mps")
else:
    DEVICE = torch.device("cpu")

print("=" * 80)
print("CONFIGURATION")
print("=" * 80)
print("Task           : Left Hand MI vs Right Hand MI")
print("Runs           :", RUNS)
print("Targets        :", [f"S{s:03d}" for s in TEST_SUBJECTS])
print("Epochs         :", NUM_EPOCHS)
print("Batch size     :", BATCH_SIZE)
print("Learning rate  :", LR)
print("Domain max     :", DOMAIN_WEIGHT_MAX)
print("SupCon weight  :", SUPCON_WEIGHT)
print("Device         :", DEVICE)
print("Results folder :", RESULTS_DIR.resolve())

CONFIGURATION
Task           : Left Hand MI vs Right Hand MI
Runs           : [4, 8, 12]
Targets        : ['S004', 'S015', 'S023', 'S029', 'S031', 'S042', 'S055', 'S071', 'S082', 'S095']
Epochs         : 120
Batch size     : 64
Learning rate  : 0.0003
Domain max     : 0.08
SupCon weight  : 0.05
Device         : mps
Results folder : /Users/ashokvarmabevara/Project2/results_binary_exact10_v2


## CELL 03 — Preflight checks

In [3]:
# ============================================================
# CELL 03 — PREFLIGHT
# ============================================================

def discover_available_subjects(data_dir, max_subjects=TOTAL_SUBJECTS):
    root = Path(data_dir)
    return [s for s in range(1, max_subjects + 1) if (root / f"S{s:03d}").is_dir()]

available_subjects = discover_available_subjects(DATA_DIR)

print("Available subject folders:", len(available_subjects))

missing_targets = [s for s in TEST_SUBJECTS if s not in available_subjects]
if missing_targets:
    raise FileNotFoundError(
        "Missing requested targets: " + ", ".join(f"S{s:03d}" for s in missing_targets)
    )

missing_runs = []
for s in TEST_SUBJECTS:
    folder = Path(DATA_DIR) / f"S{s:03d}"
    for r in RUNS:
        f = folder / f"S{s:03d}R{r:02d}.edf"
        if not f.exists():
            missing_runs.append(str(f))

if missing_runs:
    print("[WARNING] Missing target EDF files:")
    for f in missing_runs:
        print("  ", f)
else:
    print("[OK] All requested target EDF files exist for runs 4, 8, 12.")

print("[OK] Preflight complete.")

Available subject folders: 109
[OK] All requested target EDF files exist for runs 4, 8, 12.
[OK] Preflight complete.


## CELL 04 — Cached EEGMMIDB binary dataset

This is the major runtime fix. The EDF files are read and epoched **once**. Each fold then creates lightweight subject-indexed views of the cached NumPy arrays instead of rereading ~100 EDF folders for every fold.

In [4]:
# ============================================================
# CELL 04 — CACHED EEGMMIDB BINARY DATASET
# ============================================================

class EEGMMIDB_Cache:

    def __init__(self, data_dir, subjects, runs=RUNS, tmin=TMIN, tmax=TMAX, fs=FS, n_channels=N_CHANNELS):
        self.data_dir = str(data_dir)
        self.subjects = list(subjects)
        self.runs = list(runs)
        self.tmin = tmin
        self.tmax = tmax
        self.fs = fs
        self.n_channels = n_channels

        samples = []
        labels = []
        subject_ids = []
        subject_numbers = []
        run_ids = []

        start = time.time()
        loaded_runs = 0

        for subject in self.subjects:
            subject_folder = Path(self.data_dir) / f"S{subject:03d}"
            if not subject_folder.is_dir():
                continue

            for run in self.runs:
                edf = subject_folder / f"S{subject:03d}R{run:02d}.edf"
                if not edf.exists():
                    continue

                try:
                    raw = mne.io.read_raw_edf(str(edf), preload=True, verbose=False)

                    if len(raw.ch_names) < self.n_channels:
                        print(f"[WARN] S{subject:03d} R{run:02d}: fewer than {self.n_channels} channels")
                        continue

                    raw.pick(raw.ch_names[:self.n_channels])
                    raw.resample(self.fs, npad="auto")

                    events, event_id = mne.events_from_annotations(raw, verbose=False)
                    t1 = self._find_event_code(event_id, "T1")
                    t2 = self._find_event_code(event_id, "T2")

                    if t1 is None or t2 is None:
                        print(f"[WARN] S{subject:03d} R{run:02d}: T1/T2 not found")
                        continue

                    epochs = mne.Epochs(
                        raw,
                        events,
                        event_id={"T1": t1, "T2": t2},
                        tmin=self.tmin,
                        tmax=self.tmax - 1.0 / self.fs,
                        baseline=None,
                        preload=True,
                        reject_by_annotation=True,
                        verbose=False
                    )

                    data = epochs.get_data(copy=True)
                    codes = epochs.events[:, -1]

                    for i in range(len(data)):
                        code = int(codes[i])
                        if code == t1:
                            label = 0
                        elif code == t2:
                            label = 1
                        else:
                            continue

                        trial = data[i].astype(np.float32)
                        mean = trial.mean(axis=1, keepdims=True)
                        std = trial.std(axis=1, keepdims=True)
                        trial = (trial - mean) / (std + 1e-6)

                        samples.append(trial)
                        labels.append(label)
                        subject_ids.append(subject - 1)
                        subject_numbers.append(subject)
                        run_ids.append(run)

                    loaded_runs += 1

                except Exception as exc:
                    print(f"[WARN] S{subject:03d} R{run:02d}: {type(exc).__name__}: {exc}")

        if samples:
            self.X = np.stack(samples).astype(np.float32)
            self.y = np.asarray(labels, dtype=np.int64)
            self.subject_id = np.asarray(subject_ids, dtype=np.int64)
            self.subject_number = np.asarray(subject_numbers, dtype=np.int64)
            self.run_id = np.asarray(run_ids, dtype=np.int64)
        else:
            self.X = np.empty((0, self.n_channels, int(self.fs * (self.tmax - self.tmin))), dtype=np.float32)
            self.y = np.empty((0,), dtype=np.int64)
            self.subject_id = np.empty((0,), dtype=np.int64)
            self.subject_number = np.empty((0,), dtype=np.int64)
            self.run_id = np.empty((0,), dtype=np.int64)

        print(f"[CACHE] Loaded {len(self.X)} binary trials from {loaded_runs} runs in {(time.time()-start)/60:.2f} min")
        print(f"[CACHE] X shape: {self.X.shape} | memory: {self.X.nbytes/(1024**3):.3f} GB")

    @staticmethod
    def _find_event_code(event_id_dict, target_name):
        target = str(target_name).strip().upper()
        for description, code in event_id_dict.items():
            desc = str(description).strip().upper()
            if desc == target:
                return int(code)
            if desc.startswith(target):
                rest = desc[len(target):]
                if rest == "" or rest.startswith(("/", "-", "_")) or rest.isspace():
                    return int(code)
        return None

    def subset(self, subjects):
        wanted = np.asarray(list(subjects), dtype=np.int64)
        mask = np.isin(self.subject_number, wanted)
        return EEGMMIDB_Subset(self, np.flatnonzero(mask))


class EEGMMIDB_Subset(Dataset):

    def __init__(self, cache, indices):
        self.cache = cache
        self.indices = np.asarray(indices, dtype=np.int64)
        self.labels = self.cache.y[self.indices]
        self.subjects = self.cache.subject_number[self.indices]

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        j = int(self.indices[idx])
        x = torch.from_numpy(self.cache.X[j])
        y = torch.tensor(int(self.cache.y[j]), dtype=torch.long)
        s = torch.tensor(int(self.cache.subject_id[j]), dtype=torch.long)
        return x, y, s


print("Building EEG cache once...")
EEG_CACHE = EEGMMIDB_Cache(DATA_DIR, available_subjects)

if len(EEG_CACHE.X) == 0:
    raise RuntimeError("No binary EEG trials were loaded. Check DATA_DIR and EDF files.")

unique, counts = np.unique(EEG_CACHE.y, return_counts=True)
print("Global class counts:", {CLASS_NAMES[int(c)]: int(n) for c, n in zip(unique, counts)})
print("[OK] EEG cache ready.")

Building EEG cache once...
[CACHE] Loaded 4917 binary trials from 327 runs in 0.12 min
[CACHE] X shape: (4917, 22, 1000) | memory: 0.403 GB
Global class counts: {'Left Hand MI': 2479, 'Right Hand MI': 2438}
[OK] EEG cache ready.


## CELL 05 — Target-subject sanity check

In [5]:
# ============================================================
# CELL 05 — TARGET SANITY CHECK
# ============================================================

rows = []
for s in TEST_SUBJECTS:
    ds = EEG_CACHE.subset([s])
    labels = ds.labels
    u, c = np.unique(labels, return_counts=True)
    d = {int(k): int(v) for k, v in zip(u, c)}
    rows.append({
        "subject": s,
        "trials": len(ds),
        "left": d.get(0, 0),
        "right": d.get(1, 0),
        "both_classes": (0 in d and 1 in d)
    })

target_check_df = pd.DataFrame(rows)
print(target_check_df.to_string(index=False))

bad = target_check_df.loc[~target_check_df["both_classes"], "subject"].tolist()
if bad:
    raise RuntimeError("Targets missing one binary class: " + ", ".join(f"S{s:03d}" for s in bad))

print("[OK] All requested targets contain Left and Right trials.")

 subject  trials  left  right  both_classes
       4      45    23     22          True
      15      45    23     22          True
      23      45    22     23          True
      29      45    23     22          True
      31      45    22     23          True
      42      45    22     23          True
      55      45    23     22          True
      71      45    22     23          True
      82      45    23     22          True
      95      45    23     22          True
[OK] All requested targets contain Left and Right trials.


## CELL 06 — Gradient reversal + learnable Sinc filter bank

In [6]:
# ============================================================
# CELL 06 — GRL + SINC FILTER BANK
# ============================================================

class GradientReversalLayer(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, lambda_grl):
        ctx.lambda_grl = float(lambda_grl)
        return x.view_as(x)

    @staticmethod
    def backward(ctx, grad_output):
        return -ctx.lambda_grl * grad_output, None


def grl(x, lambda_grl=1.0):
    return GradientReversalLayer.apply(x, lambda_grl)


class SincFilterBank(nn.Module):
    def __init__(self, in_channels=N_CHANNELS, num_filters=NUM_FILTERS, kernel_size=SINC_KERNEL, sample_rate=FS):
        super().__init__()
        if kernel_size % 2 == 0:
            raise ValueError("SINC_KERNEL must be odd.")
        self.in_channels = in_channels
        self.num_filters = num_filters
        self.kernel_size = kernel_size
        self.sample_rate = sample_rate
        self.low_raw = nn.Parameter(torch.rand(num_filters) * 8.0 + 4.0)
        self.band_raw = nn.Parameter(torch.rand(num_filters) * 12.0 + 8.0)

    def forward(self, x):
        B, C, T = x.shape
        if C != self.in_channels:
            raise ValueError(f"SincFilterBank expected {self.in_channels} channels, got {C}")

        n = torch.arange(
            -(self.kernel_size // 2),
            self.kernel_size // 2 + 1,
            device=x.device, dtype=x.dtype
        )

        nyquist = self.sample_rate / 2.0
        filters = []

        for i in range(self.num_filters):
            low = (F.softplus(self.low_raw[i]) + 1.0).clamp(1.0, nyquist - 6.0)
            high = (low + F.softplus(self.band_raw[i]) + 2.0).clamp(3.0, nyquist - 1.0)
            high = torch.maximum(high, low + 1.0).clamp(max=nyquist - 1.0)

            f1 = low / self.sample_rate
            f2 = high / self.sample_rate

            w = (
                2.0 * f2 * torch.sinc(2.0 * f2 * n)
                -
                2.0 * f1 * torch.sinc(2.0 * f1 * n)
            )

            window = (
                0.54
                -
                0.46 * torch.cos(
                    2.0 * math.pi * (n + self.kernel_size // 2) / (self.kernel_size - 1)
                )
            )

            w = w * window
            w = w / (w.abs().sum() + 1e-6)
            filters.append(w[None, None])

        filters = torch.cat(filters, dim=0)
        z = x.reshape(B * C, 1, T)
        z = F.conv1d(z, filters, padding="same")
        z = z.reshape(B, C, self.num_filters, T)
        return z.permute(0, 2, 1, 3)

print("[OK] GRL and SincFilterBank defined.")

[OK] GRL and SincFilterBank defined.


## CELL 07 — Dynamic graph neural network

In [7]:
# ============================================================
# CELL 07 — DGNN
# ============================================================

class DGNN(nn.Module):
    def __init__(self, num_filters=NUM_FILTERS, in_nodes=N_CHANNELS, out_nodes=SPATIAL_DIM):
        super().__init__()
        self.num_filters = num_filters
        self.in_nodes = in_nodes

        self.W_Q = nn.Linear(num_filters, num_filters)
        self.W_K = nn.Linear(num_filters, num_filters)
        self.W_V = nn.Linear(in_nodes, out_nodes)

    def forward(self, x):
        B, F_band, C, T = x.shape
        if F_band != self.num_filters or C != self.in_nodes:
            raise ValueError(f"DGNN expected [B,{self.num_filters},{self.in_nodes},T], got {tuple(x.shape)}")

        descriptor = x.mean(dim=-1).transpose(1, 2)
        Q = self.W_Q(descriptor)
        K = self.W_K(descriptor)

        A = torch.matmul(Q, K.transpose(-2, -1)) / math.sqrt(self.num_filters)
        A = F.softmax(A, dim=-1)

        eye = torch.eye(C, device=x.device, dtype=x.dtype)[None]
        A = A + eye

        degree = A.sum(dim=-1).clamp_min(1e-6)
        D = torch.diag_embed(torch.rsqrt(degree))
        A_norm = D @ A @ D

        x_t = x.permute(0, 1, 3, 2)
        out = torch.einsum("bij,bftj->bfti", A_norm, x_t)
        out = self.W_V(out)
        out = F.elu(out)

        return out.permute(0, 1, 3, 2)

print("[OK] DGNN defined.")

[OK] DGNN defined.


## CELL 08 — Multi-scale temporal encoder, BiGRU, SE attention

In [8]:
# ============================================================
# CELL 08 — TEMPORAL MODULES
# ============================================================

class MultiScaleTemporalEncoder(nn.Module):
    def __init__(self, channels=SPATIAL_DIM):
        super().__init__()

        self.branches = nn.ModuleList([
            nn.Sequential(
                nn.Conv1d(channels, channels, 3, stride=2, padding=1, groups=channels, bias=False),
                nn.BatchNorm1d(channels),
                nn.ELU(inplace=True)
            ),
            nn.Sequential(
                nn.Conv1d(channels, channels, 7, stride=2, padding=3, groups=channels, bias=False),
                nn.BatchNorm1d(channels),
                nn.ELU(inplace=True)
            ),
            nn.Sequential(
                nn.Conv1d(channels, channels, 15, stride=2, padding=7, groups=channels, bias=False),
                nn.BatchNorm1d(channels),
                nn.ELU(inplace=True)
            )
        ])

        self.fuse = nn.Sequential(
            nn.Conv1d(channels * 3, channels, 1, bias=False),
            nn.BatchNorm1d(channels),
            nn.ELU(inplace=True),
            nn.AvgPool1d(2, stride=2)
        )

    def forward(self, x):
        y = torch.cat([b(x) for b in self.branches], dim=1)
        return self.fuse(y)


class SimplifiedBiMamba(nn.Module):
    """Compatibility name retained from supplied implementation; actual block is BiGRU."""
    def __init__(self, d_model=SPATIAL_DIM):
        super().__init__()
        self.ssm = nn.GRU(
            input_size=d_model,
            hidden_size=d_model // 2,
            batch_first=True,
            bidirectional=True
        )

    def forward(self, x):
        y, _ = self.ssm(x.transpose(1, 2))
        return y.transpose(1, 2)


class SEAttention(nn.Module):
    def __init__(self, channel=SPATIAL_DIM, reduction=16):
        super().__init__()
        hidden = max(1, channel // reduction)
        self.fc = nn.Sequential(
            nn.Linear(channel, hidden, bias=False),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, channel, bias=False),
            nn.Sigmoid()
        )

    def forward(self, x):
        B, C, T = x.shape
        y = x.mean(dim=2)
        w = self.fc(y).view(B, C, 1)
        return (x * w).mean(dim=2)

print("[OK] Temporal encoder, SimplifiedBiMamba, and SEAttention defined.")

[OK] Temporal encoder, SimplifiedBiMamba, and SEAttention defined.


## CELL 09 — Complete binary S³ model

In [9]:
# ============================================================
# CELL 09 — COMPLETE BINARY MODEL
# ============================================================

class S3MambaDA(nn.Module):

    def __init__(self, num_classes=N_CLASSES, num_subjects=DOMAIN_CLASSES):
        super().__init__()

        self.sinc_filter = SincFilterBank()
        self.dgnn = DGNN()
        self.temporal = MultiScaleTemporalEncoder()
        self.mamba = SimplifiedBiMamba()
        self.se_attention = SEAttention()

        self.classifier = nn.Sequential(
            nn.BatchNorm1d(SPATIAL_DIM),
            nn.Dropout(0.20),
            nn.Linear(SPATIAL_DIM, num_classes)
        )

        self.domain_classifier = nn.Sequential(
            nn.Linear(SPATIAL_DIM, 64),
            nn.ReLU(inplace=True),
            nn.Dropout(0.20),
            nn.Linear(64, num_subjects)
        )

        self.supcon_proj = nn.Sequential(
            nn.Linear(SPATIAL_DIM, 128),
            nn.ReLU(inplace=True),
            nn.Dropout(0.10),
            nn.Linear(128, 128)
        )

    def forward(self, x, lambda_grl=0.0):
        z = self.sinc_filter(x)
        z = self.dgnn(z)
        z = z.mean(dim=1)
        z = self.temporal(z)
        z = self.mamba(z)
        z = self.se_attention(z)

        class_logits = self.classifier(z)
        domain_logits = self.domain_classifier(grl(z, lambda_grl))
        z_proj = F.normalize(self.supcon_proj(z), p=2, dim=1)

        return class_logits, domain_logits, z_proj

print("[OK] S3MambaDA defined.")

[OK] S3MambaDA defined.


## CELL 10 — Stable SupCon loss + AdaBN

In [10]:
# ============================================================
# CELL 10 — LOSSES + ADABN
# ============================================================

class SupConLoss(nn.Module):
    def __init__(self, temperature=SUPCON_TEMP):
        super().__init__()
        self.temperature = temperature

    def forward(self, features, labels):
        labels = labels.view(-1, 1)
        mask = torch.eq(labels, labels.T).float()

        logits = (features @ features.T) / self.temperature
        logits = logits - logits.max(dim=1, keepdim=True).values.detach()

        logits_mask = 1.0 - torch.eye(
            features.size(0),
            device=features.device
        )

        mask = mask * logits_mask
        exp_logits = torch.exp(logits) * logits_mask
        log_prob = logits - torch.log(exp_logits.sum(dim=1, keepdim=True) + 1e-8)

        positives = mask.sum(dim=1)
        valid = positives > 0

        if valid.any():
            mean_log_prob_pos = (mask * log_prob).sum(dim=1) / (positives + 1e-8)
            return -mean_log_prob_pos[valid].mean()

        return features.sum() * 0.0


@torch.no_grad()
def apply_adabn(model, loader, device, adaptation_trials=20):
    batches = []
    seen = 0

    for x, _, _ in loader:
        batches.append(x)
        seen += x.size(0)
        if seen >= adaptation_trials:
            break

    if not batches:
        return model

    target_x = torch.cat(batches, dim=0)[:adaptation_trials].to(device)

    bn_modules = [
        m for m in model.modules()
        if isinstance(m, nn.modules.batchnorm._BatchNorm)
    ]

    if not bn_modules:
        return model

    saved = [(m.training, m.momentum) for m in bn_modules]

    for m in bn_modules:
        m.reset_running_stats()
        m.momentum = 1.0
        m.train()

    _ = model(target_x, lambda_grl=0.0)

    for m, (was_training, old_momentum) in zip(bn_modules, saved):
        m.momentum = old_momentum
        m.train(was_training)

    model.eval()
    return model

print("[OK] Stable SupCon loss and AdaBN defined.")

[OK] Stable SupCon loss and AdaBN defined.


## CELL 11 — Smoke test

This cell must pass before training.

In [11]:
# ============================================================
# CELL 11 — SMOKE TEST
# ============================================================

model_test = S3MambaDA().to(DEVICE)
x_test = torch.randn(2, N_CHANNELS, int(FS * TMAX), device=DEVICE)

with torch.no_grad():
    class_logits, domain_logits, z_proj = model_test(x_test, lambda_grl=0.0)

print("Input         :", tuple(x_test.shape))
print("Class logits  :", tuple(class_logits.shape))
print("Domain logits :", tuple(domain_logits.shape))
print("Projection    :", tuple(z_proj.shape))
print("Parameters    :", f"{sum(p.numel() for p in model_test.parameters()):,}")

assert class_logits.shape == (2, 2)
assert domain_logits.shape == (2, DOMAIN_CLASSES)
assert z_proj.shape == (2, 128)

del model_test, x_test, class_logits, domain_logits, z_proj
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()

print("[OK] Smoke test passed.")

Input         : (2, 22, 1000)
Class logits  : (2, 2)
Domain logits : (2, 109)
Projection    : (2, 128)
Parameters    : 71,775
[OK] Smoke test passed.


## CELL 12 — Subject-level validation split + balanced sampler

In [12]:
# ============================================================
# CELL 12 — VALIDATION + SAMPLER
# ============================================================

def make_subject_validation_split(subjects, val_fraction=VAL_FRACTION, seed=VAL_SEED):
    subjects = sorted(list(subjects))
    rng = np.random.RandomState(seed)
    shuffled = subjects.copy()
    rng.shuffle(shuffled)

    n_val = max(1, int(round(len(shuffled) * val_fraction)))
    val_subjects = sorted(shuffled[:n_val])
    train_subjects = sorted(shuffled[n_val:])
    return train_subjects, val_subjects


def make_balanced_sampler(dataset):
    labels = np.asarray(dataset.labels, dtype=np.int64)
    classes, counts = np.unique(labels, return_counts=True)
    if len(classes) < 2:
        raise RuntimeError("Training dataset contains only one class.")

    weights = {int(c): 1.0 / float(n) for c, n in zip(classes, counts)}
    sample_weights = np.asarray([weights[int(y)] for y in labels], dtype=np.float64)

    return WeightedRandomSampler(
        torch.tensor(sample_weights, dtype=torch.double),
        num_samples=len(sample_weights),
        replacement=True
    )

print("[OK] Validation split and balanced sampler ready.")

[OK] Validation split and balanced sampler ready.


## CELL 13 — EEG augmentation

In [13]:
# ============================================================
# CELL 13 — AUGMENTATION
# ============================================================

def augment_eeg(x):
    x = x.clone()

    scale = 1.0 + AMPLITUDE_JITTER * torch.randn(x.size(0), 1, 1, device=x.device)
    x = x * scale

    x = x + NOISE_STD * torch.randn_like(x)

    keep = (
        torch.rand(x.size(0), x.size(1), 1, device=x.device)
        > CHANNEL_DROPOUT
    )

    return x * keep

print("[OK] Augmentation ready.")

[OK] Augmentation ready.


## CELL 14 — Gradual GRL/domain schedule

In [14]:
# ============================================================
# CELL 14 — GRL / DOMAIN SCHEDULE
# ============================================================

def get_domain_weight(epoch, total_epochs):
    if epoch < DOMAIN_WARMUP_EPOCHS:
        return 0.0

    p = (epoch - DOMAIN_WARMUP_EPOCHS) / max(1, total_epochs - DOMAIN_WARMUP_EPOCHS)
    p = float(np.clip(p, 0.0, 1.0))
    return float(DOMAIN_WEIGHT_MAX * 0.5 * (1.0 - np.cos(np.pi * p)))


def get_grl_lambda(epoch, total_epochs):
    if epoch < GRL_START_EPOCH:
        return 0.0

    p = (epoch - GRL_START_EPOCH) / max(1, total_epochs - GRL_START_EPOCH)
    p = float(np.clip(p, 0.0, 1.0))
    return float(2.0 / (1.0 + np.exp(-10.0 * p)) - 1.0)

for e in [0, 9, 24, 49, 79, 119]:
    print(f"Epoch {e+1:03d} | DomainWeight={get_domain_weight(e, NUM_EPOCHS):.4f} | GRL={get_grl_lambda(e, NUM_EPOCHS):.4f}")

Epoch 001 | DomainWeight=0.0000 | GRL=0.0000
Epoch 010 | DomainWeight=0.0000 | GRL=0.0000
Epoch 025 | DomainWeight=0.0000 | GRL=0.5624
Epoch 050 | DomainWeight=0.0120 | GRL=0.9439
Epoch 080 | DomainWeight=0.0485 | GRL=0.9962
Epoch 120 | DomainWeight=0.0800 | GRL=0.9999


## CELL 15 — Evaluation helper

In [15]:
# ============================================================
# CELL 15 — EVALUATION
# ============================================================

@torch.no_grad()
def evaluate_binary(model, loader, device):
    model.eval()
    y_true, y_pred, y_prob = [], [], []

    for x, y, _ in loader:
        x = x.to(device)
        logits, _, _ = model(x, lambda_grl=0.0)
        prob = torch.softmax(logits, dim=1)[:, 1]
        pred = (prob >= 0.5).long()

        y_true.extend(y.numpy().tolist())
        y_pred.extend(pred.cpu().numpy().tolist())
        y_prob.extend(prob.cpu().numpy().tolist())

    y_true = np.asarray(y_true, dtype=np.int64)
    y_pred = np.asarray(y_pred, dtype=np.int64)
    y_prob = np.asarray(y_prob, dtype=np.float64)

    out = {
        "accuracy": accuracy_score(y_true, y_pred),
        "balanced_accuracy": balanced_accuracy_score(y_true, y_pred),
        "kappa": cohen_kappa_score(y_true, y_pred),
        "precision_macro": precision_score(y_true, y_pred, average="macro", zero_division=0),
        "recall_macro": recall_score(y_true, y_pred, average="macro", zero_division=0),
        "f1_macro": f1_score(y_true, y_pred, average="macro", zero_division=0),
        "y_true": y_true,
        "y_pred": y_pred,
        "y_prob": y_prob
    }

    try:
        out["roc_auc"] = roc_auc_score(y_true, y_prob)
    except Exception:
        out["roc_auc"] = np.nan

    return out


def selection_score(metrics):
    auc_value = 0.5 if np.isnan(metrics["roc_auc"]) else metrics["roc_auc"]
    return float(
        0.60 * metrics["balanced_accuracy"]
        +
        0.25 * auc_value
        +
        0.15 * metrics["kappa"]
    )

print("[OK] Evaluation helper ready.")

[OK] Evaluation helper ready.


## CELL 16 — Optimized training function

Validation subjects select the checkpoint. The held-out target subject is not inspected until final evaluation.

In [16]:
# ============================================================
# CELL 16 — TRAIN ONE FOLD
# ============================================================

def train_one_fold(train_dataset, val_dataset, fold_idx, num_epochs=NUM_EPOCHS):

    train_loader = DataLoader(
        train_dataset,
        batch_size=BATCH_SIZE,
        sampler=make_balanced_sampler(train_dataset),
        num_workers=0,
        drop_last=True
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=0
    )

    seed_everything(SEED + fold_idx)

    model = S3MambaDA().to(DEVICE)

    criterion_cls = nn.CrossEntropyLoss(label_smoothing=LABEL_SMOOTHING)
    criterion_domain = nn.CrossEntropyLoss()
    criterion_supcon = SupConLoss(SUPCON_TEMP)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY
    )

    warmup_epochs = 8

    def lr_factor(epoch):
        if epoch < warmup_epochs:
            return 0.15 + 0.85 * (epoch + 1) / warmup_epochs
        p = (epoch - warmup_epochs) / max(1, num_epochs - warmup_epochs)
        return MIN_LR / LR + 0.5 * (1.0 - MIN_LR / LR) * (1.0 + np.cos(np.pi * p))

    scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_factor)

    use_amp = DEVICE.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    best_score = -np.inf
    best_state = None
    patience = 0
    history = []

    for epoch in range(num_epochs):
        model.train()
        total_sum = cls_sum = dom_sum = con_sum = 0.0
        seen = 0

        domain_weight = get_domain_weight(epoch, num_epochs)
        grl_lambda = get_grl_lambda(epoch, num_epochs)

        for x, y, subject in train_loader:
            x = x.to(DEVICE)
            y = y.to(DEVICE)
            subject = subject.to(DEVICE)

            x_input = augment_eeg(x) if np.random.rand() < AUGMENT_PROB else x

            optimizer.zero_grad(set_to_none=True)

            with torch.autocast(device_type=DEVICE.type, enabled=use_amp):
                class_logits, domain_logits, z_proj = model(x_input, lambda_grl=grl_lambda)

                loss_cls = criterion_cls(class_logits, y)
                loss_domain = criterion_domain(domain_logits, subject)
                loss_supcon = criterion_supcon(z_proj, y)

                loss = (
                    loss_cls
                    +
                    domain_weight * loss_domain
                    +
                    SUPCON_WEIGHT * loss_supcon
                )

            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            scaler.step(optimizer)
            scaler.update()

            bs = x.size(0)
            seen += bs
            total_sum += loss.item() * bs
            cls_sum += loss_cls.item() * bs
            dom_sum += loss_domain.item() * bs
            con_sum += loss_supcon.item() * bs

        scheduler.step()

        avg_total = total_sum / max(1, seen)
        avg_cls = cls_sum / max(1, seen)
        avg_dom = dom_sum / max(1, seen)
        avg_con = con_sum / max(1, seen)

        val = evaluate_binary(model, val_loader, DEVICE)
        score = selection_score(val)

        history.append({
            "fold": fold_idx,
            "epoch": epoch + 1,
            "train_loss": avg_total,
            "classification_loss": avg_cls,
            "domain_loss": avg_dom,
            "supcon_loss": avg_con,
            "domain_weight": domain_weight,
            "grl_lambda": grl_lambda,
            "learning_rate": optimizer.param_groups[0]["lr"],
            "val_accuracy": val["accuracy"],
            "val_balanced_accuracy": val["balanced_accuracy"],
            "val_kappa": val["kappa"],
            "val_auc": val["roc_auc"],
            "selection_score": score
        })

        if score > best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1

        if epoch == 0 or (epoch + 1) % 10 == 0:
            print(
                f"Epoch {epoch+1:03d}/{num_epochs} | "
                f"Loss={avg_total:.4f} | "
                f"Cls={avg_cls:.4f} | "
                f"ValAcc={val['accuracy']*100:.2f}% | "
                f"ValBal={val['balanced_accuracy']*100:.2f}% | "
                f"ValAUC={val['roc_auc']:.4f}"
            )

        if epoch + 1 >= MIN_EPOCHS and patience >= PATIENCE:
            print(f"[EARLY STOP] epoch {epoch+1}")
            break

    if best_state is None:
        raise RuntimeError(f"No checkpoint was saved for fold {fold_idx}")

    model.load_state_dict(best_state)
    return model, pd.DataFrame(history), best_score

## CELL 17 — Exact ten-subject experiment

In [17]:
# ============================================================
# CELL 17 — EXACT TEN-SUBJECT EXPERIMENT
# ============================================================

def run_exact_experiment():

    fold_rows = []
    prediction_rows = []
    history_rows = []

    experiment_start = time.time()

    for fold_idx, test_subject in enumerate(TEST_SUBJECTS, start=1):

        print()
        print("=" * 90)
        print(f"FOLD {fold_idx}/{len(TEST_SUBJECTS)} | TARGET S{test_subject:03d}")
        print("=" * 90)

        source_subjects = [s for s in available_subjects if s != test_subject]

        train_subjects, val_subjects = make_subject_validation_split(
            source_subjects,
            VAL_FRACTION,
            VAL_SEED + fold_idx
        )

        train_ds = EEG_CACHE.subset(train_subjects)
        val_ds = EEG_CACHE.subset(val_subjects)
        test_ds = EEG_CACHE.subset([test_subject])

        print(
            f"Subjects | Train={len(train_subjects)} "
            f"| Val={len(val_subjects)} "
            f"| Test=1"
        )

        print(
            f"Trials   | Train={len(train_ds)} "
            f"| Val={len(val_ds)} "
            f"| Test={len(test_ds)}"
        )

        if len(np.unique(test_ds.labels)) != 2:
            raise RuntimeError(f"S{test_subject:03d} does not contain both classes")

        model, history_df, best_score = train_one_fold(
            train_ds,
            val_ds,
            fold_idx,
            NUM_EPOCHS
        )

        history_df["test_subject"] = test_subject
        history_rows.append(history_df)

        target_loader = DataLoader(
            test_ds,
            batch_size=BATCH_SIZE,
            shuffle=False,
            num_workers=0
        )

        # Label-free target-statistics adaptation.
        model = apply_adabn(
            model,
            target_loader,
            DEVICE,
            adaptation_trials=min(20, len(test_ds))
        )

        result = evaluate_binary(
            model,
            target_loader,
            DEVICE
        )

        fold_rows.append({
            "fold": fold_idx,
            "test_subject": test_subject,
            "n_train_subjects": len(train_subjects),
            "n_val_subjects": len(val_subjects),
            "n_train_trials": len(train_ds),
            "n_val_trials": len(val_ds),
            "n_test_trials": len(test_ds),
            "accuracy": result["accuracy"],
            "balanced_accuracy": result["balanced_accuracy"],
            "kappa": result["kappa"],
            "roc_auc": result["roc_auc"],
            "precision_macro": result["precision_macro"],
            "recall_macro": result["recall_macro"],
            "f1_macro": result["f1_macro"],
            "best_validation_score": best_score
        })

        for i in range(len(result["y_true"])):
            prediction_rows.append({
                "fold": fold_idx,
                "test_subject": test_subject,
                "true_label": int(result["y_true"][i]),
                "pred_label": int(result["y_pred"][i]),
                "prob_left": float(1.0 - result["y_prob"][i]),
                "prob_right": float(result["y_prob"][i])
            })

        cm = confusion_matrix(result["y_true"], result["y_pred"], labels=[0, 1])

        print()
        print(
            f"S{test_subject:03d} | "
            f"Accuracy={result['accuracy']*100:.2f}% | "
            f"BalancedAcc={result['balanced_accuracy']*100:.2f}% | "
            f"Kappa={result['kappa']:.4f} | "
            f"ROC-AUC={result['roc_auc']:.4f}"
        )
        print("Confusion matrix:")
        print(cm)

        del model, train_ds, val_ds, test_ds, target_loader
        gc.collect()
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

    fold_df = pd.DataFrame(fold_rows)
    prediction_df = pd.DataFrame(prediction_rows)
    history_df = pd.concat(history_rows, ignore_index=True)

    pooled_accuracy = accuracy_score(
        prediction_df["true_label"],
        prediction_df["pred_label"]
    )

    summary = {
        "task": "Binary Left Hand MI vs Right Hand MI",
        "runs": RUNS,
        "test_subjects": TEST_SUBJECTS,
        "completed_folds": int(len(fold_df)),
        "total_test_trials": int(fold_df["n_test_trials"].sum()),
        "pooled_accuracy": float(pooled_accuracy),
        "mean_fold_accuracy": float(fold_df["accuracy"].mean()),
        "std_fold_accuracy": float(fold_df["accuracy"].std(ddof=0)),
        "mean_balanced_accuracy": float(fold_df["balanced_accuracy"].mean()),
        "mean_kappa": float(fold_df["kappa"].mean()),
        "mean_roc_auc": float(fold_df["roc_auc"].mean()),
        "chance_accuracy": 0.50,
        "device": str(DEVICE),
        "elapsed_minutes": float((time.time() - experiment_start) / 60.0)
    }

    fold_df.to_csv(RESULTS_DIR / "fold_metrics.csv", index=False)
    prediction_df.to_csv(RESULTS_DIR / "test_predictions.csv", index=False)
    history_df.to_csv(RESULTS_DIR / "training_history.csv", index=False)

    with open(RESULTS_DIR / "summary.json", "w") as f:
        json.dump(summary, f, indent=2)

    print()
    print("=" * 90)
    print("FINAL BINARY LEFT-vs-RIGHT RESULTS")
    print("=" * 90)
    print("Targets:", ", ".join(f"S{s:03d}" for s in TEST_SUBJECTS))
    print(f"Pooled Accuracy       : {pooled_accuracy*100:.2f}%")
    print(f"Mean Fold Accuracy    : {summary['mean_fold_accuracy']*100:.2f}% ± {summary['std_fold_accuracy']*100:.2f}%")
    print(f"Mean Balanced Accuracy: {summary['mean_balanced_accuracy']*100:.2f}%")
    print(f"Mean Cohen's Kappa    : {summary['mean_kappa']:.4f}")
    print(f"Mean ROC-AUC          : {summary['mean_roc_auc']:.4f}")
    print("Chance level          : 50.00%")
    print()
    print(fold_df[[
        "test_subject", "n_test_trials", "accuracy",
        "balanced_accuracy", "kappa", "roc_auc"
    ]].to_string(index=False))
    print()
    print("Saved results to:", RESULTS_DIR.resolve())

    return fold_df, prediction_df, history_df, summary

print("[OK] Exact experiment function ready.")

[OK] Exact experiment function ready.


## CELL 18 — Optional pilot run

The pilot is **disabled by default**. It uses S004 as a target only to verify training/evaluation end-to-end without launching all ten folds.

In [18]:
# ============================================================
# CELL 18 — OPTIONAL PILOT
# ============================================================

RUN_PILOT = False

if RUN_PILOT:
    pilot = TEST_SUBJECTS[0]
    source = [s for s in available_subjects if s != pilot]
    train_subjects, val_subjects = make_subject_validation_split(source, VAL_FRACTION, VAL_SEED + 999)

    train_ds = EEG_CACHE.subset(train_subjects)
    val_ds = EEG_CACHE.subset(val_subjects)
    test_ds = EEG_CACHE.subset([pilot])

    model, pilot_history, _ = train_one_fold(train_ds, val_ds, fold_idx=999, num_epochs=40)

    loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)
    model = apply_adabn(model, loader, DEVICE, adaptation_trials=min(20, len(test_ds)))
    pilot_result = evaluate_binary(model, loader, DEVICE)

    print(
        f"Pilot S{pilot:03d}: "
        f"Accuracy={pilot_result['accuracy']*100:.2f}% | "
        f"BalancedAcc={pilot_result['balanced_accuracy']*100:.2f}% | "
        f"AUC={pilot_result['roc_auc']:.4f}"
    )

    del model, train_ds, val_ds, test_ds, loader
    gc.collect()
else:
    print("Pilot disabled. Set RUN_PILOT=True to run it.")

Pilot disabled. Set RUN_PILOT=True to run it.


## CELL 19 — Full ten-subject run

Set `RUN_FULL_EXPERIMENT = True` only after the smoke test passes.

In [22]:
# ============================================================
# CELL 19 — FULL RUN
# ============================================================

RUN_FULL_EXPERIMENT = True

if RUN_FULL_EXPERIMENT:
    fold_df, prediction_df, history_df, summary = run_exact_experiment()
else:
    print("Full experiment disabled. Set RUN_FULL_EXPERIMENT=True to launch the ten folds.")


FOLD 1/10 | TARGET S004
Subjects | Train=97 | Val=11 | Test=1
Trials   | Train=4377 | Val=495 | Test=45
Epoch 001/120 | Loss=0.9160 | Cls=0.7085 | ValAcc=50.30% | ValBal=50.00% | ValAUC=0.5753
Epoch 010/120 | Loss=0.8525 | Cls=0.6455 | ValAcc=64.24% | ValBal=64.13% | ValAUC=0.7263
Epoch 020/120 | Loss=0.8191 | Cls=0.6124 | ValAcc=68.28% | ValBal=68.22% | ValAUC=0.7427
Epoch 030/120 | Loss=0.7948 | Cls=0.5869 | ValAcc=64.44% | ValBal=64.39% | ValAUC=0.7211
[EARLY STOP] epoch 33

S004 | Accuracy=55.56% | BalancedAcc=55.53% | Kappa=0.1107 | ROC-AUC=0.6206
Confusion matrix:
[[13 10]
 [10 12]]

FOLD 2/10 | TARGET S015
Subjects | Train=97 | Val=11 | Test=1
Trials   | Train=4377 | Val=495 | Test=45
Epoch 001/120 | Loss=0.9266 | Cls=0.7191 | ValAcc=55.56% | ValBal=56.11% | ValAUC=0.5755


KeyboardInterrupt: 

## CELL 20 — Research figures

In [ ]:
# ============================================================
# CELL 20 — FIGURES
# ============================================================

def generate_figures(fold_df, prediction_df, history_df):
    if fold_df is None or fold_df.empty:
        print("No results available.")
        return

    # 1. Fold accuracy
    fig, ax = plt.subplots(figsize=(10, 5))
    labels = [f"S{int(s):03d}" for s in fold_df["test_subject"]]
    ax.bar(labels, fold_df["accuracy"] * 100)
    ax.axhline(50, linestyle="--", linewidth=1.5, label="Chance")
    ax.set_ylabel("Accuracy (%)")
    ax.set_xlabel("Held-out subject")
    ax.set_title("Binary Subject-Independent Left-vs-Right Accuracy")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig1_FoldAccuracy.png", dpi=400, bbox_inches="tight")
    plt.close(fig)

    # 2. Confusion matrix
    cm = confusion_matrix(
        prediction_df["true_label"],
        prediction_df["pred_label"],
        labels=[0, 1]
    )
    cm_norm = cm / np.maximum(cm.sum(axis=1, keepdims=True), 1)

    fig, ax = plt.subplots(figsize=(6, 5))
    im = ax.imshow(cm_norm, interpolation="nearest")
    ax.set_title("Normalized Binary Confusion Matrix")
    ax.set_xlabel("Predicted class")
    ax.set_ylabel("True class")
    ax.set_xticks([0, 1], ["Left", "Right"])
    ax.set_yticks([0, 1], ["Left", "Right"])
    for i in range(2):
        for j in range(2):
            ax.text(j, i, f"{100*cm_norm[i,j]:.1f}%", ha="center", va="center")
    fig.colorbar(im, ax=ax)
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig2_ConfusionMatrix.png", dpi=400, bbox_inches="tight")
    plt.close(fig)

    # 3. Class metrics
    report = classification_report(
        prediction_df["true_label"],
        prediction_df["pred_label"],
        labels=[0, 1],
        target_names=["Left", "Right"],
        output_dict=True,
        zero_division=0
    )
    x = np.arange(2)
    width = 0.25
    precision = [report[n]["precision"]*100 for n in ["Left", "Right"]]
    recall = [report[n]["recall"]*100 for n in ["Left", "Right"]]
    f1 = [report[n]["f1-score"]*100 for n in ["Left", "Right"]]

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.bar(x-width, precision, width, label="Precision")
    ax.bar(x, recall, width, label="Recall")
    ax.bar(x+width, f1, width, label="F1")
    ax.set_xticks(x, ["Left", "Right"])
    ax.set_ylim(0, 100)
    ax.set_ylabel("Score (%)")
    ax.set_title("Binary Per-Class Metrics")
    ax.grid(axis="y", alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig3_ClassMetrics.png", dpi=400, bbox_inches="tight")
    plt.close(fig)

    # 4. ROC
    y_true = prediction_df["true_label"].to_numpy()
    y_prob = prediction_df["prob_right"].to_numpy()
    fig, ax = plt.subplots(figsize=(7, 6))
    if len(np.unique(y_true)) == 2:
        fpr, tpr, _ = roc_curve(y_true, y_prob)
        roc_auc = auc(fpr, tpr)
        ax.plot(fpr, tpr, linewidth=2, label=f"AUC={roc_auc:.3f}")
    ax.plot([0, 1], [0, 1], linestyle="--", linewidth=1)
    ax.set_xlabel("False Positive Rate")
    ax.set_ylabel("True Positive Rate")
    ax.set_title("Binary ROC Curve")
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(FIG_DIR / "Fig4_ROC.png", dpi=400, bbox_inches="tight")
    plt.close(fig)

    # 5. Training curves
    if history_df is not None and not history_df.empty:
        g = history_df.groupby("epoch").agg({
            "train_loss":"mean",
            "classification_loss":"mean",
            "domain_loss":"mean",
            "supcon_loss":"mean",
            "val_balanced_accuracy":"mean"
        }).reset_index()

        fig, ax = plt.subplots(figsize=(8, 5))
        ax.plot(g["epoch"], g["train_loss"], label="Total loss", linewidth=2)
        ax.plot(g["epoch"], g["classification_loss"], label="Classification")
        ax.plot(g["epoch"], g["domain_loss"], label="Domain")
        ax.plot(g["epoch"], g["supcon_loss"], label="SupCon")
        ax.set_xlabel("Epoch")
        ax.set_ylabel("Loss")
        ax.set_title("Training Loss Components")
        ax.grid(True, alpha=0.25)
        ax.legend()
        fig.tight_layout()
        fig.savefig(FIG_DIR / "Fig5_TrainingLoss.png", dpi=400, bbox_inches="tight")
        plt.close(fig)

    print("Figures saved to:", FIG_DIR.resolve())

if "fold_df" in globals() and "prediction_df" in globals() and "history_df" in globals():
    generate_figures(fold_df, prediction_df, history_df)
else:
    print("Run Cell 19 first to create figures.")

## CELL 21 — Paper-ready text export

In [ ]:
# ============================================================
# CELL 21 — PAPER RESULTS EXPORT
# ============================================================

def export_paper_results(fold_df, summary):
    if fold_df is None or fold_df.empty:
        print("No results available.")
        return

    text = f"""BINARY LEFT-vs-RIGHT MOTOR-IMAGERY RESULTS
==========================================
Completed folds: {len(fold_df)}
Test subjects: {', '.join(f'S{s:03d}' for s in TEST_SUBJECTS)}

Pooled Accuracy: {summary['pooled_accuracy']*100:.2f}%
Mean Fold Accuracy: {summary['mean_fold_accuracy']*100:.2f}% ± {summary['std_fold_accuracy']*100:.2f}%
Mean Balanced Accuracy: {summary['mean_balanced_accuracy']*100:.2f}%
Mean Cohen's Kappa: {summary['mean_kappa']:.4f}
Mean ROC-AUC: {summary['mean_roc_auc']:.4f}
Binary chance level: 50.00%

Per-fold results:
{fold_df[['test_subject','accuracy','balanced_accuracy','kappa','roc_auc']].to_string(index=False)}
""".strip()

    output = RESULTS_DIR / "paper_results.txt"
    output.write_text(text)
    print(text)
    print("\nSaved:", output.resolve())

if "fold_df" in globals() and "summary" in globals():
    export_paper_results(fold_df, summary)
else:
    print("Run Cell 19 first.")

## CELL 22 — Execution order

After **Kernel → Restart Kernel**, use:

**01 → 02 → 03 → 04 → 05 → 06 → 07 → 08 → 09 → 10 → 11 → 12 → 13 → 14 → 15 → 16 → 17**

Then optionally run Cell 18. For the full evaluation set `RUN_FULL_EXPERIMENT = True` in Cell 19.

The final evaluation is fixed to the ten requested subjects and reports binary accuracy, balanced accuracy, Cohen's kappa, ROC-AUC, confusion matrix, class metrics, and training history.

**Why this notebook should be faster:** EDF loading is cached once in Cell 04; folds reuse cached NumPy arrays instead of rereading all source EDF files.

**Why this notebook is safer scientifically:** model selection uses only source-subject validation; target labels are not used for checkpoint or hyperparameter selection.